# Структура проекта и окружение Python

Когда программа растёт, её удобнее разделить на несколько файлов. А чтобы разные проекты не мешали пакетам друг друга, для каждого проекта можно создать своё виртуальное окружение.

В этом уроке разберём структуру папок, импорты, пакеты, `venv` и `pip`, а затем соберём маленький проект из нескольких модулей.

> Примеры с `%%writefile` используют команду Jupyter и записывают учебные файлы в текущую папку блокнота. Команды установки пакетов выполняй в терминале только при необходимости.

## Карта урока

1. Зачем делить программу на файлы
2. Импортируем функции из соседнего модуля
3. Защищаем точку входа через `if __name__ == "__main__"`
4. Пакет и запуск через `python -m`
5. Виртуальное окружение `venv`
6. Устанавливаем и сохраняем зависимости через `pip`
7. Практика: небольшой проект


## 1. Структура проекта

Небольшой проект — это папка с кодом и файлами, которые относятся к одной программе. Например, функции можно хранить отдельно от главной части программы.

![Структура проекта и связь между импортирующими файлами](assets/project_import.png)

Старайся не называть свои файлы именами стандартных модулей. Например, файл `json.py` рядом с программой может помешать команде `import json` найти стандартную библиотеку.


## 2. Импорт из соседнего файла

Создадим файл `lesson_tools.py` с функцией. `%%writefile` — это команда Jupyter, которая сохраняет текст ячейки в файл.


In [ ]:
%%writefile lesson_tools.py
def greet(name):
    return f"Привет, {name}!"


def make_line(symbol, count):
    return symbol * count


Теперь Python может импортировать функцию по имени файла (без `.py`). Оба файла должны находиться в доступной для импорта папке; в этом упражнении они лежат рядом с блокнотом.


In [ ]:
from lesson_tools import greet, make_line

print(greet("Аня"))
print(make_line("-", 12))

Можно импортировать модуль целиком и вызывать функции через его имя:


In [ ]:
import lesson_tools

print(lesson_tools.greet("Борис"))

Если используешь `from module import name`, в коде будут доступны только перечисленные имена. Если импортируешь модуль через `import module`, имя модуля помогает понять, откуда взялась функция.

### Импорт и повторный запуск ячеек

В интерактивном блокноте Python сохраняет уже импортированные модули в памяти. Если ты изменил `lesson_tools.py`, перезапусти ядро и снова запусти ячейки или используй `importlib.reload()`.


### Импорт выполняет команды модуля

При первом обычном импорте Python выполняет верхний уровень модуля: определения функций создают функции, а обычные `print()` и `input()` запускаются сразу. Поэтому ввод и запуск программы помещают в `main()` под проверкой `if __name__ == "__main__":`.

После изменения модуля повторный `import` обычно берёт его из кеша. Для учебного эксперимента перезапусти ядро блокнота; простого повторного запуска ячейки импорта может быть недостаточно.

**Проверка.** Добавь временный `print("Модуль загружен")` в свой модуль и импортируй его дважды. После перезапуска ядра сообщение появится снова. Затем убери отладочную печать.


In [ ]:
import importlib
import lesson_tools

importlib.reload(lesson_tools)
print(lesson_tools.greet("Вера"))

## 3. Точка входа программы

Файл можно запустить напрямую или импортировать из другого файла. Проверка `if __name__ == "__main__":` запускает нижний блок только при прямом запуске файла.

Создадим файл, который можно запускать и импортировать.


In [ ]:
%%writefile lesson_main.py
from lesson_tools import greet


def main():
    print(greet("Мир Python"))


if __name__ == "__main__":
    main()


При прямом запуске `lesson_main.py` значение `__name__` равно `"__main__"`, и функция `main()` вызывается. При импорте из другого файла условие ложно — программа не запускается сама.

Из терминала команда выглядит так:


```text
python lesson_main.py
```

На некоторых компьютерах команду Python 3 запускают как `python3 lesson_main.py`. Выбери команду, которая соответствует установленному Python.


Проверим импорт `lesson_main`: функция будет доступна, но приветствие само не напечатается.


In [ ]:
import lesson_main

print("Импорт завершён; main() автоматически не запускалась.")

## 4. Пакеты и запуск через `python -m`

Пакет помогает сгруппировать связанные модули. Для учебного проекта можно создать такую структуру:

```text
quiz_project/
├── quiz/
│   ├── __init__.py
│   ├── main.py
│   └── questions.py
└── README.md
```

Файл `__init__.py` показывает, что папка `quiz` задумана как пакет. В простом учебном проекте можно оставить его пустым.


Чтобы запустить модуль `main.py` внутри пакета из папки `quiz_project`, используй команду:

```text
python -m quiz.main
```

Запускай её из корня проекта — папки, внутри которой лежит `quiz`. Команда `-m` запускает модуль как часть пакета, поэтому импорты внутри него работают предсказуемее.


Внутри `quiz/main.py` можно импортировать функцию из соседнего модуля пакета так:

```python
from .questions import get_question
```

Точка в начале — относительный импорт: модуль ищется в том же пакете. Если запускаешь пакетный модуль, используй `python -m quiz.main`, а не запускай файл `quiz/main.py` напрямую.


### Мини-практика: разделим викторину

1. В `questions.py` создай функцию, которая возвращает вопрос и ответ.
2. В `main.py` импортируй её.
3. Вызови `main()` только в блоке `if __name__ == "__main__":`.

Пока можно потренироваться на двух файлах рядом, как в примере выше. Структуру пакета попробуй создать в отдельной папке проекта.


## 5. Виртуальное окружение `venv`

Виртуальное окружение — отдельная папка с интерпретатором и установленными пакетами для одного проекта. Это позволяет проектам использовать разные версии сторонних библиотек.

Открой терминал в папке проекта и создай окружение:

```text
python -m venv .venv
```

Если команда `python` не запускает Python 3, попробуй `python3 -m venv .venv`.


### Активируем окружение

В macOS или Linux:

```text
source .venv/bin/activate
```

В Windows PowerShell:

```text
.venv\Scripts\Activate.ps1
```

В Windows Command Prompt:

```text
.venv\Scripts\activate.bat
```

После активации в начале строки терминала обычно появляется имя окружения, например `(.venv)`. Чтобы выйти из него, введи `deactivate`.


Для проверки, какие Python и `pip` используются в терминале, запусти:

```text
python --version
python -m pip --version
```

У команды `python -m pip` есть преимущество: она запускает `pip`, связанный именно с тем интерпретатором `python`, который будет запускать проект.


Обычно виртуальное окружение не добавляют в Git. Создай файл `.gitignore` в корне проекта и добавь туда:

```text
.venv/
__pycache__/
```

`__pycache__` содержит служебные файлы Python, которые он создаёт автоматически.


## 6. Устанавливаем и сохраняем зависимости

`pip` устанавливает внешние пакеты. Сначала активируй окружение, затем установи пакет командой:

```text
python -m pip install requests
```

`requests` здесь приведён как пример стороннего пакета. Для примеров в этом блокноте он не нужен, поэтому устанавливать его не обязательно.


Посмотреть сведения об установленном пакете можно командой:

```text
python -m pip show requests
```

Показать список установленных пакетов:

```text
python -m pip list
```


Чтобы записать текущие зависимости проекта в `requirements.txt`, выполни:

```text
python -m pip freeze > requirements.txt
```

Другой человек сможет установить эти версии в своём активном окружении:

```text
python -m pip install -r requirements.txt
```

В маленьком учебном проекте можно сначала написать `requirements.txt` вручную. Не добавляй в него библиотеки, которые проект не использует.


## 7. Небольшой проект из двух модулей

Создадим мини-проект викторины. В этом примере блокнот записывает файл-модуль с функцией, а затем импортирует её.


In [ ]:
%%writefile quiz_tools.py
def check_answer(answer, correct_answer):
    return answer.strip().lower() == correct_answer.strip().lower()


Теперь используем функцию из нового файла:


In [ ]:
from quiz_tools import check_answer

answer = input("Сколько будет 6 × 7? ")
if check_answer(answer, "42"):
    print("Верно!")
else:
    print("Попробуй ещё раз.")

В настоящем проекте эти файлы лежали бы рядом в папке проекта, а не создавались бы ячейками блокнота. Когда кода станет больше, вынеси разные части программы в отдельные модули и добавь короткий `README.md` с командой запуска.


## Короткое повторение

- Модуль — файл `.py`, который можно импортировать.
- Пакет — папка, объединяющая связанные модули.
- `if __name__ == "__main__":` защищает точку входа при импорте.
- `python -m package.module` запускает модуль из корня проекта как часть пакета.
- `venv` создаёт изолированное окружение для зависимостей проекта.
- Активируй окружение перед установкой пакетов.
- `python -m pip install ...` связывает `pip` с выбранным Python.
- `requirements.txt` помогает повторно установить зависимости.

### Проверь себя

1. Зачем делить программу на модули?
2. Что произойдёт с блоком `if __name__ == "__main__":` при импорте?
3. Почему активированное окружение помогает проектам не мешать друг другу?
4. Для чего нужен `requirements.txt`?
5. Из какой папки запускать `python -m quiz.main`?


## Домашняя работа

Все условия и ячейки для решений находятся в этом блокноте. Обязательны задачи 1–3 и задача 4 на повторение. Остальные — по выбору вместе с преподавателем; не нужно выполнять весь банк за один вечер.

Для программ со стандартным вводом не добавляй приглашения к input и лишние подписи к ответу. Для функций таблицы описывают аргументы и возвращаемое значение; input нужен только там, где он указан. Для файлов и проектов таблицы описывают содержимое файлов, команды и ожидаемые результаты.

В таблицах примеров разные строки внутри ячейки означают разные строки данных. «Пустая строка (Enter)» и «Нет вывода» — поясняющие отметки, их не нужно вводить или печатать. Каждый пример проверяй отдельным запуском. Приведи ещё один собственный пример до написания кода.


### ДЗ-18.1. Общий инструмент текста · обязательная

**Сюжет.** Две программы должны пользоваться одной функцией обработки текста.

**Условие.** Создайте text_tools.py с upper_text(text), возвращающей текст в верхнем регистре. В main.py прочитайте строку, импортируйте функцию и выведите результат.

**Формат данных**

| Входные данные | Выходные данные |
|---|---|
| stdin: одна строка. Оба файла находятся в одной папке; запуск python main.py из неё. | Одна строка; при импорте text_tools постороннего вывода нет. |

**Ограничения.** До 80 символов; Python 3; сторонних пакетов нет.

**Примеры**

| Входные данные | Выходные данные |
|---|---|
| <pre>кот</pre> | <pre>КОТ</pre> |
| <pre>PyThOn</pre> | <pre>PYTHON</pre> |

**Источник.** Авторская задача курса.


### ДЗ-18.2. Запуск и импорт · обязательная

**Сюжет.** Вспомогательный модуль не должен запускать диалог при импорте.

**Условие.** В greeting.py определите main(), печатающую «Готово». Вызывайте её только под защитой __name__. При обычном import greeting вывода нет.

**Формат данных**

| Входные данные | Выходные данные |
|---|---|
| Одна из команд в таблице; запуск из папки файла. | Результат команды в stdout. |

**Ограничения.** Никакого stdin и установки пакетов не требуется.

**Примеры**

| Входные данные | Выходные данные |
|---|---|
| <pre>python greeting.py</pre> | <pre>Готово</pre> |
| <pre>python -c "import greeting"</pre> | <em>Нет вывода</em> |

**Источник.** Авторская задача курса.


### ДЗ-18.3. Изолированная мастерская · обязательная

**Сюжет.** Для учебного проекта нужно отдельное окружение, чтобы зависимости не смешивались.

**Условие.** Создайте .venv через venv. Его интерпретатором проверьте версию Python и pip. В check_env.py выведите, различаются ли sys.prefix и sys.base_prefix. Справка: после import sys поле sys.prefix содержит путь текущего интерпретатора, а sys.base_prefix — базового, на основе которого создано окружение. Сохраните команды запуска в Markdown-ячейке.

**Формат данных**

| Входные данные | Выходные данные |
|---|---|
| Рабочая папка и выбранный локальный Python; стандартный ввод не используется. | Папка .venv; вывод проверки из неё — True. Версии Python/pip фиксируются фактически установленными, универсального номера нет. |

**Ограничения.** Не помещайте .venv в сдаваемый архив. Команды активации зависят от ОС; используйте руководство урока.

**Примеры**

| Входные данные | Выходные данные |
|---|---|
| <pre>.venv/bin/python check_env.py (macOS/Linux)</pre> | <pre>True</pre> |
| <pre>.venv\Scripts\python.exe check_env.py (Windows)</pre> | <pre>True</pre> |

**Источник.** Авторская задача курса.


### ДЗ-18.4. Касса из модулей · повторение

**Сюжет.** Кассир хочет менять формат вывода отдельно от расчёта.

**Условие.** В pricing.py создайте total_cost(price, count), возвращающую произведение. В main.py организуйте чтение двух чисел и печать результата; pricing.py не читает input и не печатает.

**Формат данных**

| Входные данные | Выходные данные |
|---|---|
| Два целых числа stdin: цена, затем количество. Запуск python main.py из папки проекта. | Одно целое число. |

**Ограничения.** 0 ≤ price,count ≤ 1000.

**Примеры**

| Входные данные | Выходные данные |
|---|---|
| <pre>7<br>3</pre> | <pre>21</pre> |
| <pre>0<br>4</pre> | <pre>0</pre> |

**Источник.** Авторская задача курса.


### ДЗ-18.5. Викторина как пакет · по выбору

**Сюжет.** Учебную викторину запускают из корня проекта одинаковой командой.

**Условие.** Создайте пакет quiz с __init__.py, data.py (вопрос и ответ 4), logic.py (проверка ответа) и main.py (ввод/вывод). Команда запуска: python -m quiz.main. Вопрос можно хранить, но при точной проверке печатайте только итог.

**Формат данных**

| Входные данные | Выходные данные |
|---|---|
| Одна строка ответа stdin. | «Верно» либо «Неверно». Сдайте файлы пакета и README с командой запуска. |

**Ограничения.** Ответ сравнивается после strip; правильная строка «4»; пустой ответ допустим. Импорт модулей не запускает ввод.

**Примеры**

| Входные данные | Выходные данные |
|---|---|
| <pre>4</pre> | <pre>Верно</pre> |
| <pre>5</pre> | <pre>Неверно</pre> |

**Источник.** Авторская задача курса.


### ДЗ-18.6. Воспроизводимое окружение · по выбору

**Сюжет.** Другому ученику нужно повторить окружение с уже изученным пакетом requests.

**Условие.** В отдельном окружении установите requests, сохраните зависимости через pip freeze в requirements.txt. Создайте второе окружение и установите файл через pip install -r. В обоих средах проверьте импорт и совпадение requests.__version__.

**Формат данных**

| Входные данные | Выходные данные |
|---|---|
| Команды терминала и requirements.txt, созданный первым окружением; stdin отсутствует. | Два успешных импорта и одинаковая версия requests. В таблице критерии проверки, а не буквальный вывод с выдуманным номером версии. |

**Ограничения.** Задание дополнительное; для установки нужен доступ к индексу пакетов. Используйте python -m pip соответствующего окружения. Версию не задавайте наугад — сохраните установленную.

**Примеры**

| Входные данные | Выходные данные |
|---|---|
| <pre>Импорт requests в первом окружении</pre> | <pre>Импорт успешен; номер версии записан в отчёте</pre> |
| <pre>Импорт requests во втором окружении после установки requirements.txt</pre> | <pre>Импорт успешен; версия равна версии первого окружения</pre> |

**Источник.** Авторская задача курса.


**Что сдать:** этот блокнот с решениями и сохранёнными результатами проверок. Если условие требует модули, файлы или тесты, приложи их к работе; не прикладывай виртуальное окружение. Для каждого решения добавь свой пример и коротко объясни выбор граничного случая.
